# Decision Tree

Course 3: Supervised Learning, Klasifikasi

Decision tree menghasilkan sekumpulan aturan if-else dari akar sampai daun. Cara kerjanya mirip cara orang berpikir, sehingga mudah dibaca dan dijelaskan. Di notebook ini aku melatih pohon dengan pengaturan default, memvisualisasikannya, mengatur hyperparameter, dan mencari kombinasi terbaik dengan grid search.

## Dataset

Sama seperti notebook KNN: `tumor.csv`, hasil uji laboratorium sampel tumor, target `Class` (`0` jinak, `1` ganas):

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/tumor.csv

## 1. Import dan baca data

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RS = 123

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/tumor.csv"
tumor = pd.read_csv(URL)

X = tumor.drop(columns="Class")
y = tumor["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)

print("train:", X_train.shape, "| test:", X_test.shape)
tumor.head()

In [ ]:
def evaluasi(y_benar, y_pred):
    return {
        "accuracy": accuracy_score(y_benar, y_pred),
        "precision": precision_score(y_benar, y_pred),
        "recall": recall_score(y_benar, y_pred),
        "f1": f1_score(y_benar, y_pred),
    }

nama_fitur = X.columns.tolist()

def gambar_pohon(model, max_depth=None, ukuran=(22, 10)):
    plt.figure(figsize=ukuran)
    plot_tree(model, feature_names=nama_fitur, class_names=["jinak", "ganas"],
              filled=True, rounded=True, fontsize=8, max_depth=max_depth)
    plt.show()

## 2. Decision tree dengan pengaturan default

In [ ]:
tree_default = DecisionTreeClassifier(random_state=RS).fit(X_train, y_train)
print("kedalaman pohon :", tree_default.get_depth())
print("jumlah daun     :", tree_default.get_n_leaves())
pd.Series(evaluasi(y_test, tree_default.predict(X_test)), name="default").round(4)

Tanpa batasan apa pun, pohon dibiarkan tumbuh sampai semua data latih terklasifikasi sempurna, jadi pohonnya besar dan mudah overfit.

## 3. Visualisasi pohon

Pohon yang penuh terlalu besar untuk dibaca, jadi aku tampilkan hanya 3 tingkat pertama. Warna simpul menunjukkan kelas mayoritas di simpul itu, dan makin pekat warnanya makin murni simpulnya.

In [ ]:
gambar_pohon(tree_default, max_depth=3)

Setiap simpul menampilkan fitur dan ambang pemisah, nilai impurity (gini atau entropy), jumlah sampel, dan jumlah per kelas. Pohon yang besar bisa menghafal noise, jadi berikutnya dicoba versi yang lebih sederhana.

## 4. Mengatur hyperparameter

Tiga hyperparameter yang umum diatur:

- `criterion`: ukuran untuk memilih pemisahan, `gini` atau `entropy`
- `max_depth`: kedalaman maksimum pohon
- `min_samples_leaf`: jumlah sampel minimum di sebuah daun

In [ ]:
tree_entropy = DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_leaf=3,
                                      random_state=RS).fit(X_train, y_train)
print("kedalaman:", tree_entropy.get_depth(), "| daun:", tree_entropy.get_n_leaves())
pd.Series(evaluasi(y_test, tree_entropy.predict(X_test)), name="entropy, depth 10, leaf 3").round(4)

In [ ]:
gambar_pohon(tree_entropy, max_depth=3)

### Latihan: gini, max_depth = 15, min_samples_leaf = 5

In [ ]:
tree_gini = DecisionTreeClassifier(criterion="gini", max_depth=15, min_samples_leaf=5,
                                   random_state=RS).fit(X_train, y_train)
print("kedalaman:", tree_gini.get_depth(), "| daun:", tree_gini.get_n_leaves())
pd.Series(evaluasi(y_test, tree_gini.predict(X_test)), name="gini, depth 15, leaf 5").round(4)

In [ ]:
gambar_pohon(tree_gini, max_depth=3)

## 5. Grid search

Mencoba kombinasi satu per satu itu melelahkan. `GridSearchCV` mencoba semua kombinasi dengan cross validation 5 lipatan di data train dan memilih yang F1-nya tertinggi.

In [ ]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 10, 15, 20],
    "min_samples_leaf": [1, 2, 5, 10],
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=RS), param_grid, scoring="f1", cv=5)
grid.fit(X_train, y_train)

print("kombinasi terbaik:", grid.best_params_)
print("F1 cross validation:", round(grid.best_score_, 4))

In [ ]:
hasil_grid = pd.DataFrame(grid.cv_results_)
hasil_grid[["param_criterion", "param_max_depth", "param_min_samples_leaf", "mean_test_score"]] \
    .sort_values("mean_test_score", ascending=False).head(8).round(4)

### Latihan: pohon dengan parameter terbaik

In [ ]:
tree_terbaik = grid.best_estimator_
print("kedalaman:", tree_terbaik.get_depth(), "| daun:", tree_terbaik.get_n_leaves())
pd.Series(evaluasi(y_test, tree_terbaik.predict(X_test)), name="hasil grid search").round(4)

In [ ]:
gambar_pohon(tree_terbaik, max_depth=3)

## 6. Membaca aturan dan fitur penting

Salah satu keunggulan decision tree adalah aturannya bisa dicetak sebagai teks:

In [ ]:
print(export_text(tree_terbaik, feature_names=nama_fitur, max_depth=3))

In [ ]:
penting = pd.Series(tree_terbaik.feature_importances_, index=nama_fitur).sort_values()
penting.plot.barh(figsize=(7, 4))
plt.xlabel("pentingnya fitur (penurunan impurity)")
plt.show()
print("fitur paling penting:", ", ".join(penting.index[::-1][:3]))

## Ringkasan

- Decision tree mudah dibaca karena berupa aturan if-else, tapi pohon yang dibiarkan tumbuh penuh cenderung overfit.
- `max_depth` dan `min_samples_leaf` membatasi ukuran pohon dan umumnya memperbaiki generalisasi.
- Kombinasi hyperparameter dicari dengan `GridSearchCV` pada data train, lalu hasil akhirnya diuji di data test.
- `export_text` dan `feature_importances_` membantu menjelaskan model ke orang lain.